In [162]:
import duckdb
import pandas as pd
pd.set_option('display.max_columns', None)

con = duckdb.connect()
con.execute("""
    CREATE TABLE retail_raw AS
    SELECT * FROM read_parquet('retailcorps.parquet')
""")

In [163]:

con.execute("DESCRIBE retail_raw").df()

,column_name,column_type,null,key,default,extra
0,OrderLineID,VARCHAR,YES,None,None,None
1,Invoice,VARCHAR,YES,None,None,None
2,StockCode,VARCHAR,YES,None,None,None
3,Description,VARCHAR,YES,None,None,None
4,Quantity,VARCHAR,YES,None,None,None
5,InvoiceDate,VARCHAR,YES,None,None,None
6,CustomerID,VARCHAR,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None
8,ProductCategoryCode,VARCHAR,YES,None,None,None
9,ProductCategory,VARCHAR,YES,None,None,None


L'ensemble des valeurs sont bien en VARCHAR : conforme aux brief

In [164]:
mes_colonnes = ["CustomerID", "CustomerSegment", "CustomerType",
                "LoyaltyProgram", "Continent", "Region", "Currency"]

for col in mes_colonnes:
    print(f"===== {col} =====")
    display(con.execute(f"""
        SELECT {col} AS valeur, COUNT(*) AS nb
        FROM retail_raw
        GROUP BY 1
        ORDER BY nb DESC
        LIMIT 20
    """).df())

===== CustomerID =====


,valeur,nb
0,NaN,254201
1,17841.0,13009
2,14911.0,11532
3,12748.0,7274
4,14606.0,6674
5,14096.0,5075
6,15311.0,4682
7,14156.0,4103
8,14646.0,3853
9,13089.0,3420


===== CustomerSegment =====


,valeur,nb
0,VIP,449187
1,Loyal,227283
2,Low value,207366
3,Regular,188871


===== CustomerType =====


...

,valeur,nb
0,Europe,1067806
1,Oceania,2077
2,Asia,1617
3,North America,1033
4,South America,141
5,Africa,33


===== Region =====


,valeur,nb
0,Western Europe,1054930
1,Southern Europe,8418
2,Northern Europe,3764
3,Australia and New Zealand,2077
4,North America,1033
5,Eastern Europe,694
6,Eastern Asia,661
7,Western Asia,524
8,South-Eastern Asia,432
9,South America,141


===== Currency =====


,valeur,nb
0,GBP,988647
1,EUR,68670
2,CHF,3516
3,AUD,2369
4,NaN,1673
5,NOK,1532
6,SEK,1383
7,JPY,949
8,USD,888
9,DKK,836


CustomerID : 254 201 lignes sans clients, il s'agit de ventes réelles mais anonymes. Décision à documenter : on garde ces lignes (elles comptent dans le CA, les analyses produits/promos), mais elles seront exclues du dataset ML agrégé par client. 

Currency : 1 673 valeurs manquantes détectées à l'audit. Non corrigées : notre analyse Marketing (produits, promotions, segments) n'utilise pas la devise, les montants étant déjà exprimés en GBP (colonnes UnitPriceGBP/UnitCostGBP).

LoyaltyProgram : 4 catégories propres, mais identiques à CustomerSegment → les 2 colonnes semblent redondantes

Customer Segment : 4 catégories propres (VIP, Loyal, Low value, Regular), RAS aucune correction

CustomerType : 3 catégories propres (Retailer, Individual, Small business) RAS aucune correction

Continent : 6 valeurs cohérentes, pas de manquants RAS aucune correction

Region : 11 valeurs cohérentes, pas de manquants RAS aucune correction

In [165]:
con.execute("""
    SELECT
      SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS vrais_null,
      SUM(CASE WHEN CustomerID = 'NaN' THEN 1 ELSE 0 END) AS texte_nan,
      COUNT(DISTINCT CustomerID) AS ids_distincts,
      COUNT(*) AS total
    FROM retail_raw
""").df()

,vrais_null,texte_nan,ids_distincts,total
0,254201.0,0.0,5942,1072707


les 254 201 manquants sont de vrais NULL (pas du texte), donc pas besoin de conversion, il faut juste retirer le .0 des IDs existants. De plus, nous avons 5 942 clients identifiés sur 1 072 707 lignes.

mettre une valeur mais faire attention pour que ce soit le même par rapport au numéro de la facture 

In [166]:
con.execute("""
    SELECT CustomerSegment, LoyaltyProgram, COUNT(*) AS nb
    FROM retail_raw
    GROUP BY 1, 2
""").df()

,CustomerSegment,LoyaltyProgram,nb
0,VIP,Gold,449187
1,Regular,Bronze,188871
2,Loyal,Silver,227283
3,Low value,No,207366


Redondance confirmée : chaque segment correspond exactement à un niveau (VIP au Gold, Loyal au Silver, Regular au Bronze, Low value au No). On gardera les deux dans la dimension client (pour l'affichage Power BI) mais attention ne pas utiliser l'une comme feature pour prédire l'autre en ML car nous aurons une fuite de données. 

In [167]:
con.execute("""
    SELECT
      CASE WHEN CustomerID IS NULL THEN 'Anonyme' ELSE 'Identifié' END AS type_client,
      CustomerSegment,
      COUNT(*) AS nb,
      ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (
          PARTITION BY CASE WHEN CustomerID IS NULL THEN 'Anonyme' ELSE 'Identifié' END
      ), 1) AS pct
    FROM retail_raw
    GROUP BY 1, 2
    ORDER BY 1, nb DESC
""").df()

,type_client,CustomerSegment,nb,pct
0,Anonyme,VIP,246723,97.1
1,Anonyme,Loyal,2696,1.1
2,Anonyme,Low value,2552,1.0
3,Anonyme,Regular,2230,0.9
4,Identifié,Loyal,224587,27.4
5,Identifié,Low value,204814,25.0
6,Identifié,VIP,202464,24.7
7,Identifié,Regular,186641,22.8


Anomalie détectée :
Chez les ventes (CustomerID → '0'), 97,1% sont taguées VIP/Gold, contre seulement 24,7% chez les clients identifiés. Il peut s'agir d'une perte de données pour les clients VIP/Gold" ou une valeur par défaut assignée quand un client n'est pas identifié, plutôt qu'une vraie caractéristique.

In [168]:
con.execute("""
    SELECT
      CASE WHEN CustomerID IS NULL THEN 'Anonyme' ELSE 'Identifié' END AS type_client,
      CustomerSegment,
      AVG(TRY_CAST(Quantity AS DOUBLE))     AS qte_moyenne,
      AVG(TRY_CAST(UnitPriceGBP AS DOUBLE)) AS prix_moyen
    FROM retail_raw
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

,type_client,CustomerSegment,qte_moyenne,prix_moyen
0,Anonyme,Low value,128.912461,11.770789
1,Anonyme,Loyal,162.832588,11.851714
2,Anonyme,Regular,193.956326,11.777473
3,Anonyme,VIP,66.012819,11.443383
4,Identifié,Low value,79.395278,11.729076
5,Identifié,Loyal,67.721692,11.717611
6,Identifié,Regular,68.879458,11.773934
7,Identifié,VIP,71.432894,11.742322


ces ventes restent de vrais VIP en termes de CA/comportement, mais sans identité rattachable → toujours à exclure du dataset ML par client (pas de vrai CustomerID), mais on peut les compter dans les KPI agrégés par segment sans trop de biais.

In [169]:
con.execute("""
    SELECT Country, COUNT(*) AS nb
    FROM retail_raw
    WHERE Currency = 'NaN' OR Currency IS NULL
    GROUP BY 1
    ORDER BY nb DESC
""").df()

,Country,nb
0,United Kingdom,1506
1,France,28
2,EIRE,27
3,UNITED KINGDOM,24
4,Germany,21
5,Netherlands,16
6,Spain,8
7,Switzerland,7
8,U.K.,5
9,Denmark,4


La colonne Country est sale : Gros nettoyage à faire. créer une colonne country_clean 

NETTOYAGE

Création table client nettoyée (le REPLACE corrige le ".0")

In [170]:
con.execute("""
    CREATE OR REPLACE TABLE clean_client AS
    SELECT
        REPLACE(CustomerID, '.0', '') AS CustomerID,
        CustomerSegment,
        CustomerType,
        LoyaltyProgram
    FROM retail_raw
""")

In [171]:
con.execute("""
    UPDATE clean_client
    SET CustomerID = '0'
    WHERE CustomerID IS NULL
""")

Vérification : plus aucun ID se terminant par .0

In [172]:
con.execute("SELECT COUNT(*) AS restants FROM clean_client WHERE CustomerID LIKE '%.0'").df()

,restants
0,0


Remplacement des valeurs NULL dans CustomerID par 0 qui correspond aux clients anonymes

In [173]:
con.execute("""
    UPDATE clean_client
    SET CustomerID = '0'
    WHERE CustomerID IS NULL
""")

vérification

In [174]:
con.execute("""
    SELECT
      SUM(CASE WHEN CustomerID = '0' THEN 1 ELSE 0 END) AS clients_anonymes,
      SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS nulls_restants
    FROM clean_client
""").df()

,clients_anonymes,nulls_restants
0,254201.0,0.0


In [188]:
con.execute("""
    CREATE OR REPLACE TABLE dim_client AS
    SELECT DISTINCT CustomerID, CustomerSegment, CustomerType, LoyaltyProgram
    FROM clean_client
    WHERE CustomerID <> '0'

    UNION ALL

    SELECT '0', 'Inconnu', 'Inconnu', 'Inconnu'
""")

In [189]:
con.execute("SELECT COUNT(*) AS nb_lignes, COUNT(DISTINCT CustomerID) AS nb_clients FROM dim_client").df()

,nb_lignes,nb_clients
0,5943,5943


Création table géographie

In [175]:
con.execute("""
    CREATE OR REPLACE TABLE clean_geographie AS
    SELECT Country, Continent, Region, Currency
    FROM retail_raw
""")

Création de la table country_clean

In [176]:
con.execute("ALTER TABLE clean_geographie ADD COLUMN country_clean VARCHAR")

Nettoyage des pays

In [177]:
con.execute("""
    UPDATE clean_geographie
    SET country_clean = CASE
        WHEN UPPER(TRIM(Country)) IN ('UNITED KINGDOM', 'UK', 'U.K.', 'GREAT BRITAIN') THEN 'United Kingdom'
        WHEN UPPER(TRIM(Country)) IN ('GERMANY', 'DEUTSCHLAND', 'DE')                  THEN 'Germany'
        WHEN UPPER(TRIM(Country)) IN ('FRANCE', 'FR')                                  THEN 'France'
        WHEN UPPER(TRIM(Country)) IN ('EIRE', 'REPUBLIC OF IRELAND')                   THEN 'Ireland'
        WHEN UPPER(TRIM(Country)) IN ('NETHERLANDS', 'HOLLAND', 'NL')                  THEN 'Netherlands'
        WHEN UPPER(TRIM(Country)) IN ('SWITZERLAND', 'CH')                             THEN 'Switzerland'
        WHEN UPPER(TRIM(Country)) IN ('AUSTRALIA', 'AUS')                              THEN 'Australia'
        WHEN UPPER(TRIM(Country)) IN ('USA', 'UNITED STATES')                          THEN 'United States'
        WHEN UPPER(TRIM(Country)) = 'UNITED ARAB EMIRATES'                             THEN 'United Arab Emirates'
        WHEN UPPER(TRIM(Country)) = 'SOUTH AFRICA'                                     THEN 'South Africa'
        ELSE UPPER(SUBSTRING(TRIM(Country), 1, 1)) || LOWER(SUBSTRING(TRIM(Country), 2))
    END
""")

In [178]:
con.execute("ALTER TABLE clean_geographie DROP COLUMN Currency")

In [185]:
con.execute("ALTER TABLE clean_geographie DROP COLUMN Country")

In [186]:
con.execute("DESCRIBE clean_geographie").df()

,column_name,column_type,null,key,default,extra
0,Continent,VARCHAR,YES,None,None,None
1,Region,VARCHAR,YES,None,None,None
2,country_clean,VARCHAR,YES,None,None,None


Vérification

In [180]:
con.execute("""
    SELECT Country, country_clean, COUNT(*) AS nb
    FROM clean_geographie
    GROUP BY 1, 2
    ORDER BY country_clean, nb DESC
""").df()

,Country,country_clean,nb
0,Australia,Australia,2041
1,AUSTRALIA,Australia,22
2,AUS,Australia,8
3,australia,Australia,6
4,Belgium,Belgium,3187
...,...,...,...
56,U.K.,United Kingdom,1716
57,united kingdom,United Kingdom,1702
58,Great Britain,United Kingdom,1641
59,USA,United States,604


Vérification du nombre de pays : DISTINCT country_clean contenait 29 entrées - 7 anomalies = 22 pays

In [181]:
con.execute("SELECT COUNT(DISTINCT country_clean) AS nb_pays FROM clean_geographie").df()

,nb_pays
0,22


In [191]:
con.execute("""
    CREATE OR REPLACE TABLE dim_geographie AS
    SELECT DISTINCT country_clean, Continent, Region
    FROM clean_geographie
""")

In [192]:
con.execute("SELECT COUNT(*) FROM dim_geographie").df()

,count_star()
0,22


Création table TEMPS

trouver la période couverte par les données (pour savoir quelles dates générer)

In [182]:
con.execute("""
    SELECT
      MIN(TRY_CAST(InvoiceDate AS TIMESTAMP)) AS date_min,
      MAX(TRY_CAST(InvoiceDate AS TIMESTAMP)) AS date_max
    FROM retail_raw
""").df()

,date_min,date_max
0,2009-12-01 07:45:00,2011-12-09 12:50:00


In [183]:
con.execute("""
    CREATE OR REPLACE TABLE dim_date AS
    SELECT
        CAST(STRFTIME(d, '%Y%m%d') AS INTEGER) AS date_key,
        CAST(d AS DATE)  AS full_date,
        YEAR(d)          AS annee,
        QUARTER(d)       AS trimestre,
        MONTH(d)         AS mois,
        MONTHNAME(d)     AS nom_mois
    FROM generate_series(DATE '2009-12-01', DATE '2011-12-31', INTERVAL 1 DAY) AS t(d)
""")

j'arrondis au 31 décembre, c'est plus propre qu'un arrêt au 9 décembre (2 ans de données)

In [194]:
con.execute("DESCRIBE dim_client").df()


,column_name,column_type,null,key,default,extra
0,CustomerID,VARCHAR,YES,None,None,None
1,CustomerSegment,VARCHAR,YES,None,None,None
2,CustomerType,VARCHAR,YES,None,None,None
3,LoyaltyProgram,VARCHAR,YES,None,None,None


In [195]:
con.execute("DESCRIBE dim_geographie").df()

,column_name,column_type,null,key,default,extra
0,country_clean,VARCHAR,YES,None,None,None
1,Continent,VARCHAR,YES,None,None,None
2,Region,VARCHAR,YES,None,None,None


In [196]:
con.execute("DESCRIBE dim_date").df()

,column_name,column_type,null,key,default,extra
0,date_key,INTEGER,YES,None,None,None
1,full_date,DATE,YES,None,None,None
2,annee,BIGINT,YES,None,None,None
3,trimestre,BIGINT,YES,None,None,None
4,mois,BIGINT,YES,None,None,None
5,nom_mois,VARCHAR,YES,None,None,None


In [198]:
con.execute("COPY dim_client TO 'dim_client.parquet' (FORMAT PARQUET)")
con.execute("COPY dim_geographie TO 'dim_geographie.parquet' (FORMAT PARQUET)")
con.execute("COPY dim_date TO 'dim_date.parquet' (FORMAT PARQUET)")